In [35]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

In [36]:
file_path = "Student Mental health.csv"
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shariful07/student-mental-health",
    file_path
)

/var/folders/qp/jxy8rmnj7vj_yrr93rx2bpvm0000gn/T/ipykernel_5645/422904663.py:2: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [37]:
df.shape

(101, 11)

In [38]:
df.head()

,Timestamp,Choose your gender,Age,What is your course?,Your current year of Study,What is your CGPA?,Marital status,Do you have Depression?,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


In [39]:
df.columns = [
    'timestamp',
    'gender',
    'age',
    'course',
    'year_of_study',
    'cgpa',
    'marital_status',
    'depression',
    'anxiety',
    'panic_attack',
    'treatment'
]

In [40]:
df.head()

,timestamp,gender,age,course,year_of_study,cgpa,marital_status,depression,anxiety,panic_attack,treatment
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


In [41]:
# remove timestamp column
df = df.drop('timestamp', axis=1)

In [42]:
df.head()

,gender,age,course,year_of_study,cgpa,marital_status,depression,anxiety,panic_attack,treatment
0,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gender          101 non-null    str    
 1   age             100 non-null    float64
 2   course          101 non-null    str    
 3   year_of_study   101 non-null    str    
 4   cgpa            101 non-null    str    
 5   marital_status  101 non-null    str    
 6   depression      101 non-null    str    
 7   anxiety         101 non-null    str    
 8   panic_attack    101 non-null    str    
 9   treatment       101 non-null    str    
dtypes: float64(1), str(9)
memory usage: 8.0 KB


In [44]:
# What is the prevalence of mental health conditions?
# Convert responses to numeric 0/1 where applicable and compute mean as prevalence
cols = ['depression', 'anxiety', 'panic_attack']

# Map common string/boolean representations to binary 1/0
df[cols] = df[cols].replace({
    'Yes': 1, 'No': 0, 'yes': 1, 'no': 0, 'Y': 1, 'N': 0, 'y': 1, 'n': 0,
    'True': 1, 'False': 0, 'true': 1, 'false': 0,
    '1': 1, '0': 0
})

# Coerce any remaining values to numeric (non-convertible -> NaN)
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# Calculate prevalence (mean of 0/1 gives proportion)
prevalence = pd.DataFrame({
    'metric': cols,
    'rate': df[cols].mean().values
})

prevalence


,metric,rate
0,depression,0.346535
1,anxiety,0.336634
2,panic_attack,0.326733


In [45]:
#Is anxiety related to depression?
anxiety_vs_depression = pd.crosstab(
    df["depression"],
    df["anxiety"],
    normalize="index"
)

In [46]:
anxiety_vs_depression

anxiety,0,1
depression,,
0,0.757576,0.242424
1,0.485714,0.514286


In [47]:
#Do panic attacks increase depression risk?
panic_vs_depression = pd.crosstab(
    df["depression"],
    df["panic_attack"],
    normalize="index"
)
panic_vs_depression

panic_attack,0,1
depression,,
0,0.757576,0.242424
1,0.514286,0.485714


In [48]:
#Does gender affect depression?
gender_depression = (
    df.groupby("gender")["depression"]
    .mean()
    .rename("depression_rate")
    .reset_index()
)
gender_depression

,gender,depression_rate
0,Female,0.386667
1,Male,0.230769


In [49]:
#Does year of study influence anxiety?
year_anxiety = (
    df.groupby("year_of_study")["anxiety"]
    .mean()
    .rename("anxiety_rate")
    .reset_index()
)
year_anxiety

,year_of_study,anxiety_rate
0,Year 1,0.500000
1,Year 2,0.437500
2,Year 3,0.368421
3,year 1,0.317073
4,year 2,0.300000
5,year 3,0.200000
6,year 4,0.250000


In [50]:
#Does CGPA relate to panic attacks?
cgpa_panic = (
    df.groupby("cgpa")["panic_attack"]
    .mean()
    .rename("panic_rate")
    .reset_index()
)

In [51]:
cgpa_panic

,cgpa,panic_rate
0,0 - 1.99,0.250000
1,2.00 - 2.49,0.500000
2,2.50 - 2.99,0.750000
3,3.00 - 3.49,0.209302
4,3.50 - 4.00,0.382979
5,3.50 - 4.00,1.000000


In [52]:
#Which courses have higher depression?
course_depression = (
    df.groupby("course")["depression"]
    .mean()
    .sort_values(ascending=False)
    .rename("depression_rate")
    .reset_index()
)
course_depression

,course,depression_rate
0,ALA,1.000000
1,ENM,1.000000
2,koe,1.000000
3,Usuluddin,1.000000
4,Pendidikan islam,1.000000
5,Nursing,1.000000
6,Marine science,1.000000
7,Malcom,1.000000
8,MHSC,1.000000
9,Law,1.000000


In [54]:
# Are depressed students receiving treatment?
# Normalize treatment column to numeric 0/1 before aggregating
df['treatment'] = df['treatment'].replace({
    'Yes': 1, 'No': 0, 'yes': 1, 'no': 0, 'Y': 1, 'N': 0, 'y': 1, 'n': 0,
    'True': 1, 'False': 0, 'true': 1, 'false': 0,
    '1': 1, '0': 0
})
# Coerce any remaining values to numeric (non-convertible -> NaN)
df['treatment'] = pd.to_numeric(df['treatment'], errors='coerce')

# Group and compute counts and treated sums; fill NaN treated with 0
# Use named aggregations for clarity
treatment_gap = (
    df.groupby('depression')['treatment']
    .agg(count='count', treated='sum')
    .assign(treated=lambda x: x['treated'].fillna(0).astype(int),
            untreated=lambda x: x['count'] - x['treated'])
    .reset_index()
)

treatment_gap


,depression,count,treated,untreated
0,0,66,0,66
1,1,35,6,29


In [57]:
# What factors influence depression most?
# Preprocess features: coerce numeric, one-hot encode categoricals, fill missing values
from sklearn.ensemble import RandomForestClassifier

# Prepare X and y
X = df.drop(columns=["depression"]).copy()
y = df["depression"].astype(int)

# Convert plausible numeric columns
if 'cgpa' in X.columns:
    X['cgpa'] = pd.to_numeric(X['cgpa'], errors='coerce')

# One-hot encode categorical variables (drop_first to avoid multicollinearity)
X_encoded = pd.get_dummies(X, drop_first=True)

# Fill missing values with column medians
X_encoded = X_encoded.fillna(X_encoded.median())

# Fit model
model = RandomForestClassifier(random_state=42)
model.fit(X_encoded, y)

# Extract feature importances
feature_importance = (
    pd.Series(model.feature_importances_, index=X_encoded.columns)
    .sort_values(ascending=False)
    .rename("importance")
    .reset_index()
    .rename(columns={"index": "feature"})
)

# Show top features
feature_importance.head(20)


,feature,importance
0,marital_status_Yes,0.204718
1,age,0.119999
2,anxiety,0.069269
3,panic_attack,0.064745
4,treatment,0.058741
5,course_BENL,0.045488
6,course_Engineering,0.039226
7,gender_Male,0.034021
8,course_BIT,0.030690
9,course_BCS,0.029346


In [58]:
#What are the strongest correlations with depression?
correlation = (
    df.corr(numeric_only=True)["depression"]
    .drop("depression")
    .abs()
    .sort_values(ascending=False)
    .rename("correlation")
    .reset_index()
    .rename(columns={"index": "feature"})
)

In [59]:
correlation

,feature,correlation
0,treatment,0.345105
1,anxiety,0.273764
2,panic_attack,0.246842
3,age,0.072171


In [60]:
#Which group is most at risk?
high_risk_group = pd.pivot_table(
    df,
    values="depression",
    index="year_of_study",
    columns="gender",
    aggfunc="mean"
)
high_risk_group

gender,Female,Male
year_of_study,,
Year 1,0.000000,1.000000
Year 2,0.222222,0.428571
Year 3,0.466667,0.250000
year 1,0.406250,0.000000
year 2,0.666667,0.250000
year 3,0.400000,NaN
year 4,0.142857,0.000000
